## 1. Import Libraries 

In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

## 2. Load Data 

In [4]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_test = pd.read_csv("data/y_test.csv").values.ravel()

## 3. Group Rare Classes & Encode Labels 

In [5]:
rare_classes = ["type", "hover", "scrolldown", "appear", "disappear", "scrollup"]
y_train = pd.Series(y_train).replace(rare_classes, "other").values
y_test = pd.Series(y_test).replace(rare_classes, "other").values

classes = np.sort(np.unique(y_train))
print("Classes:", classes)
print(f"Number of classes: {len(classes)}")

# XGBoost requires integer labels for multi-class classification
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

Classes: ['click' 'drag' 'other' 'select' 'zoomin' 'zoomout']
Number of classes: 6


## 4. Build Pipeline & Hyperparameter Tuning (GridSearchCV)

In [6]:
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb", xgb.XGBClassifier(
        objective="multi:softmax",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "xgb__n_estimators": [100, 200, 300],
    "xgb__max_depth": [3, 5, 7],
    "xgb__learning_rate": [0.05, 0.1, 0.3],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train_enc)

print("Best parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred_enc = best_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc.astype(int))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'xgb__colsample_bytree': 1.0, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 3, 'xgb__n_estimators': 100, 'xgb__subsample': 0.8}
Best Score: 0.48412058814809616


## 5. Evaluate Model Performance

In [7]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Multi-class XGBoost Results:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f} (weighted)")
print(f"  Recall:    {recall:.4f} (weighted)")
print(f"  F1:        {f1:.4f} (weighted)")

print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Multi-class XGBoost Results:
  Accuracy:  0.6934
  Precision: 0.7422 (weighted)
  Recall:    0.6934 (weighted)
  F1:        0.7062 (weighted)
Classification report:
              precision    recall  f1-score   support

       click       0.65      0.71      0.68       291
        drag       0.85      0.64      0.73       426
       other       0.25      0.65      0.37        20
      select       0.25      0.57      0.34        23
      zoomin       0.84      0.90      0.87        70
     zoomout       0.70      0.84      0.76        67

    accuracy                           0.69       897
   macro avg       0.59      0.72      0.62       897
weighted avg       0.74      0.69      0.71       897

